# 02d · 복합 푸아송과 검사 역설 (Compound Poisson Processes and the Inspection Paradox)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §2.3 (복합 푸아송의 평균·분산) · Durrett 3e §3.3 (나이·잔여수명, 푸아송 특수 경우) · Ross 12e §5.4.2 (복합 푸아송 mgf, 보험 청구 예제) · Ross 12e §7.7 (검사 역설) · G&S 4e §6.8 (복합 푸아송 연습) · G&S 4e §10.3 (excess life) |
| 선수 노트북 | 02c (세분화·NHPP·PASTA), 02b (세 정의·순서통계량), 00c (조건부 기대·랜덤 합) |
| 예상 소요 | 75분 |
| 상태 | draft |

이 노트북의 두 아이디어: **(1)** 도착마다 독립 크기를 붙이면(복합 푸아송) 분산은 $\lambda t\,\mathrm{Var}(Y)$가 아니라 $\lambda t\,E[Y^2]$다 — 도착 수의 요동이 더해지기 때문이다. **(2)** 간격들은 평균 $1/\lambda$인데, '고정 시각 $t$를 덮는 간격'은 평균이 거의 $2/\lambda$다 — 긴 간격일수록 $t$에 걸리기 쉽다(길이 편향). 그래서 버스를 평균 간격의 절반이 아니라 **전체**만큼 기다린다.

## 0. 준비 (Setup)

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy import stats
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.poisson import compound_poisson, poisson_arrivals, age_residual_at
from spkit.renewal import renewal_times
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("02d")
setup_plots()
N_PATHS = scale(200_000)   # 복합 푸아송 표본 수; FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_AGE = scale(50_000)      # 나이·잔여수명 경로 수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_AGE={N_AGE}")

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def se_of_variance(x):
    """표본분산의 표준오차 sqrt((m4 - m2^2)/n) (델타 방법)."""
    x = np.asarray(x, dtype=float); n = x.size; d = x - x.mean()
    m2 = (d**2).mean(); m4 = (d**4).mean()
    return float(np.sqrt((m4 - m2**2) / n))


def compound_exp_tail_series(mu, x, n_max=200):
    """Exp(1) 점프 복합 푸아송(N ~ Poisson(mu))의 꼬리 P(X > x) = sum_n P(N=n) P(Gamma(n,1) > x)."""
    n = np.arange(1, n_max + 1)
    return float((stats.poisson(mu).pmf(n) * stats.gamma(a=n, scale=1.0).sf(x)).sum())

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (02c) 비율 $\lambda$ 푸아송 과정을 확률 $p$로 세분화하면 무엇을 얻는가?

<details><summary>정답</summary>

비율 $p\lambda$와 $(1-p)\lambda$인 **서로 독립인** 푸아송 과정 두 개. 증명: $P(N_1=j,N_2=k)$가 두 푸아송 pmf의 곱으로 갈라진다.

</details>

**Q2.** (02c) NHPP에서 $N(t)-N(s)$의 분포는? 정상 증분이 있는가?

<details><summary>정답</summary>

$\mathrm{Poisson}(\Lambda(t)-\Lambda(s))$, $\Lambda(t)=\int_0^t\lambda(u)\,du$. 정상 증분은 없다 — 분포가 구간의 길이뿐 아니라 위치에 의존한다.

</details>

**Q3.** (02c) NHPP를 세분화(thinning)로 시뮬레이션할 때 $\lambda_{\max}$의 조건과 채택확률은?

<details><summary>정답</summary>

$\lambda_{\max}\ge\sup_{t\le T}\lambda(t)$; 시각 $t$의 후보 도착을 확률 $\lambda(t)/\lambda_{\max}$로 유지한다. 조건이 깨지면 실제 비율이 $\min(\lambda(t),\lambda_{\max})$로 잘린다.

</details>

**Q4 (연결 고리).** (02b) 비율 $\lambda$ 푸아송 과정에서 $k$번째 도착시각 $T_k$의 분포와, $P(T_k\le t)$와 $N(t)$의 관계는?

<details><summary>정답</summary>

$T_k\sim\mathrm{Gamma}(k,\lambda)$; $P(T_k\le t)=P(N(t)\ge k)$ ('$k$번째 도착이 $t$ 이전' = '$t$까지 $k$개 이상'). 오늘은 이 쌍대성을 거꾸로 써서 $\{A(t)>a\}=\{(t-a,t]$에 도착 없음$\}$ 같은 사건을 푸아송 확률로 바꾼다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **복합 푸아송 과정(compound Poisson process).** $N(t)$는 비율 $\lambda$ 푸아송 과정, $Y_1,Y_2,\dots$는 iid 점프 크기(jump size)이고 $N$과 독립. $X(t)=\sum_{i=1}^{N(t)}Y_i$ ($N(t)=0$이면 $0$). 예: 보험 청구 총액, 고객당 매출의 누적, 02b 따름정리 (d)의 $\sum_{i\le N(T)}T_i$. spkit: `compound_poisson(rate, T, jump_sampler, rng, n_paths)` → $X(T)$ 표본 (`jump_sampler(n, rng)`가 점프 $n$개를 돌려준다).
- **적률생성함수(mgf).** $E[e^{\theta X(t)}]=\exp\big(\lambda t\,(M_Y(\theta)-1)\big)$, $M_Y(\theta)=E[e^{\theta Y}]$. $N$에 조건부 후 tower(00c). 로그 mgf $\lambda t(M_Y(\theta)-1)$을 미분하면 누적률(cumulant)이 전부 나온다: $E[X(t)]=\lambda tE[Y]$, $\mathrm{Var}(X(t))=\lambda tE[Y^2]$, 3차 중심적률 $=\lambda tE[Y^3]$.
- **나이(age)와 잔여수명(residual life).** 고정 시각 $t$에 대해 $A(t)=t-T_{N(t)}$ (마지막 도착 이후 경과 시간; 도착이 없었으면 $t$), $R(t)=T_{N(t)+1}-t$ (다음 도착까지). $t$를 덮는 간격 $L(t)=A(t)+R(t)=T_{N(t)+1}-T_{N(t)}$. spkit: `age_residual_at(times, t)` → `(A, R)`.
- **검사 역설(inspection paradox).** 간격들은 iid $\mathrm{Exp}(\lambda)$(평균 $1/\lambda$)인데, '고정 시각 $t$를 덮는 간격'의 평균은 $(2-e^{-\lambda t})/\lambda\to2/\lambda$로 거의 두 배다. 긴 간격일수록 $t$에 '걸릴' 확률이 길이에 비례해 크기 때문이다.
- **길이 편향 분포(size-biased distribution).** 밀도 $f$, 평균 $\mu$인 간격을 길이에 비례해 뽑으면 밀도는 $x f(x)/\mu$. 지수분포면 $x\cdot\lambda e^{-\lambda x}/(1/\lambda)=\lambda^2xe^{-\lambda x}=\mathrm{Gamma}(2,\lambda)$, 평균 $2/\lambda$. 일반 재생과정(04b)에서는 $E[L_\infty]=E[X^2]/E[X]$.
- **이 노트북의 매개변수.** 복합 푸아송: $\lambda=2$, $T=5$, $Y\sim\mathrm{Exp}(1)$ ($E[Y]=1$, $E[Y^2]=2$). 검사 역설: $\lambda=1$, $t\in\{0.25,0.5,1,2,4,8\}$; 경로는 $(0,60]$에 생성한다(4.4절의 무작위 시각 $U\le30$ 뒤에도 $30$의 여유: $P(\text{그 뒤 도착 없음})=e^{-30}\approx 10^{-13}$이라 잔여수명이 `inf`가 되는 일이 없다).

### 2.2 정리 1 — 복합 푸아송의 평균과 분산 (Durrett 3e §2.3, Ross 12e §5.4.2)

> $E[Y^2]<\infty$이면 $E[X(t)]=\lambda t\,E[Y]$, $\mathrm{Var}(X(t))=\lambda t\,E[Y^2]$.
> 일반 랜덤 합 $S=\sum_{i\le N}Y_i$에 대해서는 $\mathrm{Var}(S)=E[N]\mathrm{Var}(Y)+\mathrm{Var}(N)E[Y]^2$이고, $N\sim\mathrm{Poisson}(\lambda t)$면 $E[N]=\mathrm{Var}(N)=\lambda t$라서 $\lambda t(\mathrm{Var}(Y)+E[Y]^2)=\lambda tE[Y^2]$.

**가정이 왜 필요한가**
- $N$과 $(Y_i)$의 **독립성**: 조건부 기대 $E[S\mid N=n]=nE[Y]$를 쓰려면 필요하다. 점프 크기가 도착 수에 의존하면(예: 붐빌수록 작은 구매) 공식이 깨진다.
- $Y_i$ **iid**: $\mathrm{Var}(S\mid N=n)=n\mathrm{Var}(Y)$에 필요하다.
- $E[Y^2]<\infty$: 없으면 분산이 무한(예: 파레토 청구액, $\alpha<2$) — 보험의 '대재해' 모형이 여기서 갈린다.
- $N$이 **푸아송**이어야 $\lambda tE[Y^2]$로 단순화된다. 일반 $N$은 총분산 법칙 형태를 그대로 써야 한다.

<details><summary>증명 (완전 증명)</summary>

$S=\sum_{i\le N}Y_i$, $N$은 $(Y_i)$와 독립, $E[Y^2]<\infty$라 하자.

**평균.** 독립성 덕분에 $N=n$에 조건부로 $S$는 $n$개 iid 합이므로 $E[S\mid N=n]=nE[Y]$, 즉 $E[S\mid N]=N\,E[Y]$. tower로
$$E[S]=E\big[E[S\mid N]\big]=E[N\,E[Y]]=E[N]\,E[Y].$$

**분산 (총분산 법칙, 00c).** $\mathrm{Var}(S)=E[\mathrm{Var}(S\mid N)]+\mathrm{Var}(E[S\mid N])$. 첫 항: $\mathrm{Var}(S\mid N=n)=n\mathrm{Var}(Y)$ (iid 합의 분산), 따라서 $E[\mathrm{Var}(S\mid N)]=E[N]\mathrm{Var}(Y)$. 둘째 항: $\mathrm{Var}(N\,E[Y])=\mathrm{Var}(N)E[Y]^2$. 합하면
$$\mathrm{Var}(S)=E[N]\,\mathrm{Var}(Y)+\mathrm{Var}(N)\,E[Y]^2 .$$
$N=N(t)\sim\mathrm{Poisson}(\lambda t)$를 대입하면 $E[N]=\mathrm{Var}(N)=\lambda t$이므로 $\mathrm{Var}(X(t))=\lambda t\,(\mathrm{Var}(Y)+E[Y]^2)=\lambda t\,E[Y^2]$. $\square$

**mgf로 다시.** $E[e^{\theta S}\mid N=n]=M_Y(\theta)^n$이므로
$$E[e^{\theta X(t)}]=\sum_{n\ge0}e^{-\lambda t}\frac{(\lambda t)^n}{n!}M_Y(\theta)^n=e^{-\lambda t}e^{\lambda tM_Y(\theta)}=\exp\big(\lambda t(M_Y(\theta)-1)\big).$$
$\log$ mgf $=\lambda t(M_Y(\theta)-1)$을 $\theta=0$에서 한 번 미분하면 $\lambda tM_Y'(0)=\lambda tE[Y]$, 두 번 미분하면 $\lambda tM_Y''(0)=\lambda tE[Y^2]$ — 같은 결과다. 세 번 미분하면 3차 누적률 $\lambda tE[Y^3]$: 지수 점프면 $E[Y^3]=6>0$이라 $X(t)$의 히스토그램은 오른쪽으로 치우친다(4절 그림).

**함정 시연.** $Y\equiv1$이면 $X(t)=N(t)$이고 $\mathrm{Var}(Y)=0$이지만 $\mathrm{Var}(X(t))=\lambda t\cdot1^2=\lambda t>0$. 분산의 정체는 '점프 크기의 요동'만이 아니라 '점프 **수**의 요동'이다.

</details>

### 2.3 정리 2 — 푸아송 과정의 나이·잔여수명과 검사 역설 (Durrett 3e §3.3, G&S 4e §10.3)

> 비율 $\lambda$ 푸아송 과정, 고정 $t>0$.
> (a) $R(t)\sim\mathrm{Exp}(\lambda)$.
> (b) $A(t)\overset{d}{=}\min(\mathrm{Exp}(\lambda),t)$: $P(A(t)>a)=e^{-\lambda a}$ ($0\le a<t$), $P(A(t)=t)=e^{-\lambda t}$.
> (c) $A(t)\perp R(t)$.
> (d) $E[L(t)]=E[A(t)]+E[R(t)]=\dfrac{1-e^{-\lambda t}}{\lambda}+\dfrac1\lambda=\dfrac{2-e^{-\lambda t}}{\lambda}\xrightarrow{t\to\infty}\dfrac2\lambda$, 그리고 $L(t)\xrightarrow{d}\mathrm{Gamma}(2,\lambda)$.

**가정이 왜 필요한가**
- **지수 간격(무기억성)**: (a)는 '$t$에서 다음 도착까지'가 과거와 무관하게 $\mathrm{Exp}(\lambda)$라는 무기억성 그 자체다. 일반 재생과정에서는 $R(t)$의 극한분포가 $\bar F(x)/\mu$(평형분포, 04b)로 바뀐다.
- $t$가 **고정**(과정과 독립)이어야 한다: $t$를 과정을 보고 고르면(예: '어떤 도착 직후') 편향이 달라진다. 무작위 $U(0,T)$ 시각도 과정과 독립이면 같은 결론이다.
- **독립 증분**: (c)는 $(t-a,t]$와 $(t,t+r]$의 도착 수가 독립이라는 데서 나온다.

<details><summary>증명 (완전 증명)</summary>

핵심은 사건을 '어떤 구간에 도착이 없다'로 바꾸는 것이다. $N$의 증분은 $\mathrm{Poisson}(\lambda\cdot\text{길이})$이므로 길이 $\ell$인 구간에 도착이 없을 확률은 $e^{-\lambda\ell}$.

**(a)** $r\ge0$에 대해 $\{R(t)>r\}=\{N(t+r)-N(t)=0\}=\{(t,t+r]\text{에 도착 없음}\}$, 확률 $e^{-\lambda r}$. 이것이 $\mathrm{Exp}(\lambda)$의 생존함수다.

**(b)** $0\le a<t$에 대해 $\{A(t)>a\}=\{N(t)-N(t-a)=0\}=\{(t-a,t]\text{에 도착 없음}\}$, 확률 $e^{-\lambda a}$. $a\ge t$면 $A(t)\le t\le a$가 항상 성립하므로 $P(A(t)>a)=0$. 특히 $P(A(t)=t)=P(N(t)=0)=e^{-\lambda t}$ — 분포에 **원자(atom)** 가 있다. 이 생존함수는 정확히 $\min(E,t)$, $E\sim\mathrm{Exp}(\lambda)$의 것이다.

**(c)** $a<t$, $r\ge0$에 대해 $\{A(t)>a\}\cap\{R(t)>r\}=\{(t-a,t]\text{와 }(t,t+r]\text{ 모두 도착 없음}\}$. 두 구간은 서로소이므로 독립 증분에 의해 확률은 $e^{-\lambda a}e^{-\lambda r}=P(A(t)>a)P(R(t)>r)$. 결합 생존함수가 곱으로 갈라지므로 독립이다($a\ge t$인 경우는 자명).

**(d)** 음이 아닌 확률변수의 기댓값 공식 $E[A]=\int_0^\infty P(A>a)\,da$에서 피적분함수는 $a\ge t$일 때 $0$이므로 적분 상한이 $t$가 된다:
$$E[A(t)]=\int_0^t e^{-\lambda a}\,da=\frac{1-e^{-\lambda t}}{\lambda},\qquad E[R(t)]=\frac1\lambda,$$
$$E[L(t)]=\frac{2-e^{-\lambda t}}{\lambda}\to\frac2\lambda .$$
**극한 분포.** $t\to\infty$에서 $A(t)\xrightarrow{d}\mathrm{Exp}(\lambda)$ (원자 $e^{-\lambda t}\to0$), $R(t)\sim\mathrm{Exp}(\lambda)$, 둘은 독립이므로 $L(t)$는 독립 $\mathrm{Exp}(\lambda)$ 둘의 합 $\mathrm{Gamma}(2,\lambda)$로 수렴한다. 합성곱으로 밀도를 쓰면
$$\int_0^x\lambda e^{-\lambda a}\,\lambda e^{-\lambda(x-a)}\,da=\lambda^2xe^{-\lambda x}=\frac{x\cdot\lambda e^{-\lambda x}}{1/\lambda},$$
즉 지수분포 밀도 $f(x)=\lambda e^{-\lambda x}$를 길이 $x$에 비례해 재가중한 **길이 편향 밀도** $xf(x)/\mu$다. 이것이 '역설'의 해소다: 시각으로 고른 간격은 번호로 고른 간격이 아니라 그 길이 편향 버전이다. $\square$

</details>

### 2.4 정리 3 — 길이 편향 (04b 예고; Ross 12e §7.7, Durrett 3e §3.3)

> iid 양의 간격 $X_i$ (밀도 $f$, 평균 $\mu$, $E[X^2]<\infty$)로 만든 재생과정에서 $t$를 덮는 간격 $L(t)$는 $t\to\infty$에서 밀도 $xf(x)/\mu$로 수렴하고 $E[L_\infty]=E[X^2]/\mu\ge\mu$ (등호는 $\mathrm{Var}(X)=0$일 때만). 잔여수명의 극한 평균은 $E[X^2]/(2\mu)$.

**가정이 왜 필요한가**
- $E[X^2]<\infty$: 없으면 극한 평균이 무한이다 — 무거운 꼬리 간격에서는 '무작위 시각에 걸린 간격'이 폭발한다.
- **비격자(non-lattice)** 간격: 비격자 간격이 아니면 $L(t)$, $R(t)$의 분포가 수렴하지 않고 주기적으로 진동한다(상수 간격이면 $R(t)$가 톱니처럼 진동; $L\equiv\mu$는 자명). 평균에 대한 등식 $E[L_\infty]=E[X^2]/\mu$는 시간평균의 의미로는 격자 경우에도 성립하며, $E[X^2]/\mu=\mu$의 등호는 $\mathrm{Var}(X)=0$, 즉 상수 간격뿐이다 (E3).

**증명**: 재생-보상 정리로 증명한다(04a/04b에서). 직관은 한 줄이다 — 길이 $x$인 간격이 $[0,T]$에서 차지하는 시간은 $x$에 비례하므로, 무작위 시각이 그 간격에 떨어질 확률도 $x$에 비례한다. 지수분포에서 $E[X^2]/\mu=(2/\lambda^2)/(1/\lambda)=2/\lambda$로 정리 2와 일치한다. 여기서는 인용만 하고 E3에서 $U(0.5,1.5)$ 간격으로 수치 확인한다.

### 2.5 직관

복합 푸아송은 '도착 수는 푸아송, 도착마다 독립 크기'다. 분산이 $\lambda tE[Y^2]$인 이유는 도착 수의 요동($\mathrm{Var}(N)E[Y]^2$)과 크기의 요동($E[N]\mathrm{Var}(Y)$)이 **더해지기** 때문이다.

검사 역설은 표본추출 방식의 문제다. 간격을 '번호로' 고르면 평균 $1/\lambda$, '시각으로' 고르면 긴 간격이 더 자주 잡혀 평균 $2/\lambda$. 버스 정류장에 무작위로 도착한 사람은 평균 $2/\lambda$짜리 간격 안에 있고, 무기억성 덕에 그 간격의 '뒤쪽 절반'의 기댓값이 다시 $1/\lambda$ — 그래서 평균 간격의 절반이 아니라 **전체**만큼 기다린다. 02c의 PASTA(푸아송 도착 = 무작위 스냅샷)와 같은 눈이다: 무작위 시각에서 본 시스템은 시간 평균을 보여주며, 시간 평균은 긴 상태를 길게 본다.

### 2.6 함정

- $\mathrm{Var}(X(t))=\lambda tE[Y^2]$이지 $\lambda t\mathrm{Var}(Y)$가 아니다. 상수 점프 $Y\equiv c$여도 분산은 $\lambda tc^2$이다.
- `compound_poisson`의 `jump_sampler`는 `(n, rng) -> array`. 비율/평균 규약을 다시 확인하라 — `rng.exponential(1.0, n)`은 **평균** 1(scale 인자)이다.
- 나이 $A(t)$는 $t$에서 잘린다: $t$가 작으면 $E[A(t)]=(1-e^{-\lambda t})/\lambda<1/\lambda$이고 따라서 $E[L(t)]<2/\lambda$. '$2/\lambda$'는 극한값이다.
- 잔여수명이 $\mathrm{Exp}(\lambda)$인 것은 푸아송(무기억) 특유의 성질이다. 규칙적인 버스($U(0.5,1.5)$ 간격)는 대기 평균이 $E[X^2]/(2\mu)=13/24\approx0.54$로 간격 평균의 약 절반이다 (E3).
- 구성 B(02b, $(0,T]$ 위 균등 정렬)로 만든 경로는 $T$ 뒤 도착이 없어 `age_residual_at`의 잔여수명이 `np.inf`가 된다. 이 노트북은 구성 A를 (가장 늦은 관측 시각)$+30$까지 생성해 이를 피한다; `np.isinf` 개수가 0임을 출력으로 확인한다. 여유가 8뿐이면 $e^{-8}\approx3\times10^{-4}$ — 경로 5만 개 중 열댓 개가 `inf`가 되어 히스토그램이 터진다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

(1) $\mathrm{Var}(X(t))=\lambda tE[Y^2]$에서 왜 $\mathrm{Var}(Y)$가 아니라 $E[Y^2]$인지 상수 점프 예로 설명. (2) '버스 평균 간격 10분인데 왜 평균 10분을 기다리나'를 친구에게 세 문장으로. (3) $E[A(t)]=\int_0^te^{-\lambda a}\,da$의 적분 하한/상한이 왜 $0$과 $t$인지.

## 3. Predict — 코드를 돌리기 전에 예측

아래 값들을 계산기 없이(공식으로) 먼저 적어 보세요. 정답은 5절에서 시뮬레이션·닫힌 형식과 나란히 비교합니다.

In [ ]:
predictions = {
    # λ=2, T=5, Y~Exp(1)인 복합 푸아송 X(5)의 평균은?
    "mean_X": None,
    # Var(X(5))는? (힌트: λT·Var(Y) = 10 이 아니다)
    "var_X": None,
    # P(X(5) > 15)는? (참고값; 대략의 크기만 — 정규근사 N(10, 20)으로 어림해 보라)
    "p_X_gt_15": None,
    # λ=1 푸아송 과정에서 잔여수명 R(1)의 평균은?
    "mean_R_t1": None,
    # t=1을 덮는 간격 L(1)의 평균은? (전형적 간격의 평균은 1)
    "mean_L_t1": None,
    # 각 경로에서 U~U(0,30)인 무작위 시각을 골라 그 시각을 덮는 간격의 평균은?
    "mean_L_uniform_time": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 복합 푸아송 $X(5)$, $\lambda=2$, $Y\sim\mathrm{Exp}(1)$

`compound_poisson`은 경로마다 $N\sim\mathrm{Poisson}(\lambda T)$를 뽑고 점프 $N$개를 붙여 $X(T)$만 돌려준다(경로 전체는 필요 없다).

In [ ]:
LAM_C, T_C = 2.0, 5.0
jump = lambda n, g: g.exponential(1.0, n)          # Exp(1): 평균 1, E[Y^2] = 2
X = compound_poisson(LAM_C, T_C, jump, rng, n_paths=N_PATHS)

est_mean_X = mc_estimate(X)
var_X_hat, var_X_se = float(X.var(ddof=1)), se_of_variance(X)
est_p_gt_15 = mc_proportion(X > 15)
print(f"E[X(5)]      : {est_mean_X}")
print(f"Var(X(5))    : {var_X_hat:.4f} ± {var_X_se:.4f}")
print(f"P(X(5) > 15) : {est_p_gt_15}")
print(f"P(X(5) = 0)  : {np.mean(X == 0):.2e}  (exact e^-10 = {math.exp(-10):.2e})")

왼쪽은 경로 5개(도착시각은 `poisson_arrivals`, 점프는 지수)를 계단 함수로, 오른쪽은 $X(5)$의 히스토그램과 참고용 정규근사 $N(10,20)$를 그린다. 3차 누적률 $\lambda TE[Y^3]=60>0$이라 오른쪽 꼬리가 정규보다 두껍고, $x=0$에는 $P(X=0)=e^{-10}$의 원자가 있다(눈에는 안 보인다).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for k in range(5):
    arr = poisson_arrivals(LAM_C, T_C, rng)
    t_k = np.concatenate([[0.0], arr, [T_C]])
    x_k = np.concatenate([[0.0], np.cumsum(jump(arr.size, rng))])
    plot_paths(t_k, np.append(x_k, x_k[-1]), ax=ax1, step=True, lw=1.2)
ax1.axhline(LAM_C * T_C, color="k", ls=":", label="E[X(5)] = 10")
ax1.set_ylabel("X(t)"); ax1.set_title("5 sample paths of X(t)"); ax1.legend()

xs = np.linspace(0, 40, 300)
plot_hist_vs_pdf(X, xs, stats.norm(10, math.sqrt(20)).pdf(xs), ax=ax2, bins=60,
                 label_pdf="N(10, 20) reference")
ax2.set_xlabel("X(5)")
fig.suptitle("Compound Poisson: mean 10, variance 20 (= lambda T E[Y^2])")
plt.tight_layout(); plt.show()

### 4.2 총분산 법칙 분해와 상수 점프 함정 (참고 출력)

같은 설정을 손으로 다시 만든다: $N\sim\mathrm{Poisson}(10)$을 직접 뽑고, 정리 1의 두 항 $E[N]\mathrm{Var}(Y)$와 $\mathrm{Var}(N)E[Y]^2$를 표본으로 계산해 합이 20에 가까움을 본다(표본분산의 SE는 약 0.23(2만 표본), 0.07(20만 표본)이라 20에서 2–3 SE 어긋날 수 있다 — 여기는 참고 출력이고 5절에서 SE 기준으로 판정한다). 그리고 상수 점프 $Y\equiv1$로 돌려 분산이 $0$이 아니라 $\lambda T=10$임을 확인한다.

In [ ]:
N_ = rng.poisson(LAM_C * T_C, N_PATHS)
Y_all = jump(int(N_.sum()), rng)
S_ = np.bincount(np.repeat(np.arange(N_PATHS), N_), weights=Y_all, minlength=N_PATHS)
term_count = N_.mean() * Y_all.var(ddof=1)        # E[N] Var(Y)  (≈ 10·1)
term_size = N_.var(ddof=1) * Y_all.mean() ** 2    # Var(N) E[Y]^2 (≈ 10·1)
print(f"E[N]·Var(Y)   = {term_count:.3f}")
print(f"Var(N)·E[Y]^2 = {term_size:.3f}")
print(f"sum           = {term_count + term_size:.3f}   vs sample Var(S) = {S_.var(ddof=1):.3f} ± {se_of_variance(S_):.3f}")

X_const = compound_poisson(LAM_C, T_C, lambda n, g: np.ones(n), rng, n_paths=N_PATHS)
print(f"constant jumps Y=1: Var(X(5)) = {X_const.var(ddof=1):.3f}  (λT = {LAM_C * T_C}, not 0)")

### 4.3 나이·잔여수명: $\lambda=1$, 고정 시각 $t\in\{0.25,0.5,1,2,4,8\}$

경로를 $(0,60]$까지 만들어(4.4절에서 쓸 무작위 시각 $U\le30$ 뒤에도 30의 여유) 각 $t$에서 `age_residual_at`으로 $(A(t),R(t))$를 읽는다. 잔여수명이 `inf`인 경로가 0개임을 반드시 확인한다.

In [ ]:
LAM_A, T_GEN = 1.0, 60.0
t_list = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
paths = [poisson_arrivals(LAM_A, T_GEN, rng) for _ in range(N_AGE)]

A = np.empty((N_AGE, len(t_list))); R = np.empty((N_AGE, len(t_list)))
for j, t in enumerate(t_list):
    A[:, j], R[:, j] = np.array([age_residual_at(p, t) for p in paths]).T
L = A + R
n_inf = int(np.isinf(R).sum())
print(f"paths={N_AGE}, mean #arrivals={np.mean([p.size for p in paths]):.1f}, residual=inf count: {n_inf}")

A_t1, R_t1, A_t8, R_t8 = A[:, 2], R[:, 2], A[:, 5], R[:, 5]
print(f"t=1 : E[A]={mc_estimate(A_t1)}   E[R]={mc_estimate(R_t1)}")
print(f"t=1 : E[L]={mc_estimate(A_t1 + R_t1)}   P(A=t)={np.mean(A_t1 == 1.0):.4f} (e^-1={math.exp(-1):.4f})")
print(f"t=8 : E[L]={mc_estimate(A_t8 + R_t8)}")

그림 1: 세 평균의 MC 추정치(±2SE)를 $t$에 대해 그리고 닫힌 곡선 $1-e^{-t}$, $1$, $2-e^{-t}$를 겹친다. $E[L(t)]$는 $2$에 아래에서 접근한다.

In [ ]:
tt = np.linspace(0.01, 9, 300)
fig, ax = plt.subplots()
for arr_, lab, curve in [(A, "E[A(t)]", 1 - np.exp(-LAM_A * tt)),
                         (R, "E[R(t)]", np.ones_like(tt)),
                         (L, "E[L(t)]", 2 - np.exp(-LAM_A * tt))]:
    ests = [mc_estimate(arr_[:, j]) for j in range(len(t_list))]
    eb = ax.errorbar(t_list, [e.mean for e in ests], yerr=[2 * e.se for e in ests],
                     fmt="o", capsize=3, label=f"{lab} (MC ± 2SE)")
    ax.plot(tt, curve, color=eb[0].get_color(), lw=1.5)
ax.axhline(2, color="k", ls="--", lw=1, label="2/lambda")
ax.set_xlabel("t"); ax.set_ylabel("mean"); ax.set_xscale("log")
ax.set_title("Inspection paradox: the interval covering t is length-biased")
ax.legend(); plt.show()

그림 2: $t=8$의 $L$ 히스토그램 vs 극한 밀도 $\mathrm{Gamma}(2,1)$: $xe^{-x}$ (점선은 전형적 간격의 밀도 $e^{-x}$). 그림 3: $t=1$의 $A(1)$ — $[0,1)$ 위의 연속 부분(밀도 $e^{-a}$)과 $a=1$의 원자 $e^{-1}$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
xs = np.linspace(0, 10, 300)
plot_hist_vs_pdf(A_t8 + R_t8, xs, stats.gamma(a=2, scale=1 / LAM_A).pdf(xs), ax=ax1, bins=60,
                 label_pdf="Gamma(2,1): x e^-x (length-biased)")
ax1.plot(xs, stats.expon(scale=1 / LAM_A).pdf(xs), "k--", lw=1.2, label="Exp(1): e^-x (typical gap)")
ax1.set_xlim(0, 10); ax1.set_xlabel("L(8) = A(8) + R(8)"); ax1.legend()
ax1.set_title("Interval covering t = 8 vs typical interval")

bw = 0.025; edges = np.arange(0, 1 + bw / 2, bw)
cont = A_t1[A_t1 < 1.0]
ax2.hist(cont, bins=edges, weights=np.full(cont.size, 1 / (A_t1.size * bw)), alpha=0.5,
         label="simulation (continuous part)")
aa = np.linspace(0, 1, 200)
ax2.plot(aa, LAM_A * np.exp(-LAM_A * aa), color="C1", lw=2, label="sub-density e^-a")
ax2.bar([1.0], [np.mean(A_t1 == 1.0)], width=0.03, color="C3",
        label=f"atom at a = t: MC {np.mean(A_t1 == 1.0):.3f} vs e^-1 = {math.exp(-1):.3f}")
ax2.set_xlabel("A(1)"); ax2.set_title("Age at t = 1: truncated exponential + atom"); ax2.legend()
plt.tight_layout(); plt.show()

### 4.4 같은 과정, 두 가지 고르기: 번호로 vs 시각으로

같은 `paths`에서 (a) 각 경로의 **5번째** 간격(번호로 고르기), (b) 각 경로에서 무작위 시각 $U\sim U(0,30)$을 골라 그 시각을 덮는 간격(시각으로 고르기), (c) 그 무작위 시각의 잔여수명(버스 대기시간)을 잰다. (b)의 히스토그램에 겹친 $xe^{-x}$는 극한 밀도다; $U$가 작은 구간의 절단 효과(원자 $e^{-U}$)로 정확한 평균은 $2-1/30$이다(5절).

In [ ]:
GAP_INDEX, T_U = 4, 30.0
gap_idx = np.array([np.diff(np.concatenate([[0.0], p]))[GAP_INDEX] for p in paths])   # (a) 5번째 간격 T_5 - T_4 (첫 간격은 T_1 - 0)
u_times = rng.uniform(0.0, T_U, N_AGE)
A_u, R_u = np.array([age_residual_at(p, u) for p, u in zip(paths, u_times)]).T
L_u = A_u + R_u                                                      # (b) 무작위 시각을 덮는 간격

print(f"(a) gap chosen by index #5      : {mc_estimate(gap_idx)}   (typical gap, 1/λ = 1)")
print(f"(b) gap covering a uniform time : {mc_estimate(L_u)}   (≈ 2/λ = 2)")
print(f"(c) residual at a uniform time  : {mc_estimate(R_u)}   (bus wait = FULL mean gap, not half)")
n_inf_u = int(np.isinf(R_u).sum())
print(f"residual=inf count at random times: {n_inf_u}  (paths run to T_GEN = {T_GEN} = T_U + 30)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
xs = np.linspace(0, 10, 300)
plot_hist_vs_pdf(gap_idx, xs, stats.expon(scale=1 / LAM_A).pdf(xs), ax=ax1, bins=60,
                 label_pdf="Exp(1) pdf, mean 1")
ax1.set_xlim(0, 10); ax1.set_xlabel("5th interarrival T_5 - T_4 of each path"); ax1.set_title("Gap chosen by index")
plot_hist_vs_pdf(L_u, xs, stats.gamma(a=2, scale=1 / LAM_A).pdf(xs), ax=ax2, bins=60,
                 label_pdf="Gamma(2,1) pdf (large-t limit), mean 2")
ax2.set_xlim(0, 10); ax2.set_xlabel("gap covering U ~ U(0, 30)"); ax2.set_title("Gap chosen by time")
fig.suptitle("Same process, two sampling rules"); plt.tight_layout(); plt.show()

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 전부 공식으로 계산한다. $E[Y]$, $E[Y^2]$는 `scipy.stats.expon`의 적률에서, $P(X(5)>15)$는 급수 $\sum_n P(N=n)P(\mathrm{Gamma}(n,1)>15)$에서, 무작위 시각의 $E[L(U)]$는 $\frac1{T_u}\int_0^{T_u}\frac{2-e^{-\lambda t}}{\lambda}dt=\frac2\lambda-\frac{1-e^{-\lambda T_u}}{\lambda^2T_u}$에서 온다.

In [ ]:
EY, EY2 = stats.expon(scale=1.0).moment(1), stats.expon(scale=1.0).moment(2)
mean_X_exact = LAM_C * T_C * EY
var_X_exact = LAM_C * T_C * EY2
p_X_gt_15_series = compound_exp_tail_series(LAM_C * T_C, 15.0)
E_A = lambda t: (1 - math.exp(-LAM_A * t)) / LAM_A
E_L = lambda t: (2 - math.exp(-LAM_A * t)) / LAM_A
E_L_unif = 2 / LAM_A - (1 - math.exp(-LAM_A * T_U)) / (LAM_A**2 * T_U)

rows = [
    {"name": "mean_X", "predicted": predictions["mean_X"], "estimate": est_mean_X, "exact": mean_X_exact},
    {"name": "var_X", "predicted": predictions["var_X"], "estimate": var_X_hat, "se": var_X_se, "exact": var_X_exact},
    {"name": "p_X_gt_15", "predicted": predictions["p_X_gt_15"], "estimate": est_p_gt_15, "exact": p_X_gt_15_series},
    {"name": "mean_A_t1", "predicted": None, "estimate": mc_estimate(A_t1), "exact": E_A(1.0)},
    {"name": "mean_R_t1", "predicted": predictions["mean_R_t1"], "estimate": mc_estimate(R_t1), "exact": 1 / LAM_A},
    {"name": "mean_L_t1", "predicted": predictions["mean_L_t1"], "estimate": mc_estimate(A_t1 + R_t1), "exact": E_L(1.0)},
    {"name": "mean_L_t8", "predicted": None, "estimate": mc_estimate(A_t8 + R_t8), "exact": E_L(8.0)},
    {"name": "mean_gap_by_index", "predicted": None, "estimate": mc_estimate(gap_idx), "exact": 1 / LAM_A},
    {"name": "mean_L_uniform_time", "predicted": predictions["mean_L_uniform_time"], "estimate": mc_estimate(L_u), "exact": E_L_unif},
    {"name": "mean_R_uniform_time", "predicted": None, "estimate": mc_estimate(R_u), "exact": 1 / LAM_A},
]
Markdown(compare_table(rows))

행마다 불일치가 뜻하는 것:
- `mean_X`: $\lambda TE[Y]$가 틀렸다면 `jump_sampler`의 평균 규약(scale vs rate)이 어긋난 것이다.
- `var_X`: $\lambda TE[Y^2]$ 대신 $\lambda T\mathrm{Var}(Y)=10$ 근처가 나오면 도착 수의 요동을 빠뜨린 것이다.
- `p_X_gt_15`: 급수와 어긋나면 꼬리(3차 누적률)를 정규근사로 대체한 셈이다 — $N(10,20)$은 $0.132$를 준다.
- `mean_A_t1`: $(1-e^{-1})$과 어긋나면 나이가 $t$에서 잘린다는 사실(원자)을 놓친 것이다.
- `mean_R_t1`: $1/\lambda$가 아니면 무기억성이 깨진 것 — 간격이 지수분포가 아니다.
- `mean_L_t1`: 나이·잔여수명의 합이므로 위 둘 중 하나의 오류가 전파된 것이다.
- `mean_L_t8`: $2-e^{-8}\approx2$에서 벗어나면 극한 $2/\lambda$(정리 2 (d))이 틀린 것이다.
- `mean_gap_by_index`: 번호로 고른 간격은 그냥 $\mathrm{Exp}(\lambda)$ — 1이 아니면 경로 생성이 잘못된 것이다.
- `mean_L_uniform_time`: $2-1/30$과 어긋나면 무작위 시각이 과정과 독립이 아니거나(예: 도착 직후를 골랐거나) 경로 끝의 `inf` 절단이 섞인 것이다.
- `mean_R_uniform_time`: 1이 아니라 0.5 근처면 '대기 = 간격의 절반'이라는 잘못된 직관이 시뮬레이션에 스며든 것이다.

In [ ]:
assert n_inf == 0 and n_inf_u == 0, "residuals must be finite: paths generated to T_GEN = 60 = T_U + 30"
assert_close(est_mean_X, mean_X_exact, k=4, name="mean_X")
assert_close(var_X_hat, var_X_exact, se=var_X_se, k=4, name="var_X")
assert_close(est_p_gt_15, p_X_gt_15_series, k=4, name="p_X_gt_15")
assert_close(mc_estimate(A_t1), E_A(1.0), k=4, name="mean_A_t1")
assert_close(mc_estimate(R_t1), 1 / LAM_A, k=4, name="mean_R_t1")
assert_close(mc_estimate(A_t1 + R_t1), E_L(1.0), k=4, name="mean_L_t1")
assert_close(mc_estimate(A_t8 + R_t8), E_L(8.0), k=4, name="mean_L_t8")
assert_close(mc_estimate(gap_idx), 1 / LAM_A, k=4, name="mean_gap_by_index")
assert_close(mc_estimate(L_u), E_L_unif, k=4, name="mean_L_uniform_time")
assert_close(mc_estimate(R_u), 1 / LAM_A, k=4, name="mean_R_uniform_time")
print("all checks passed (k=4 SE)")

## 6. 연습문제

### E1 (계산)

(a) 보험사에 청구가 하루 평균 3건(푸아송)으로 들어오고 청구액 $Y$는 $E[Y]=100$, $\mathrm{Var}(Y)=2500$이다. 30일 총 청구액의 평균, 분산, 표준편차를 구하라.
(b) 버스가 시간당 4대 푸아송으로 온다. 무작위로 정류장에 도착한 사람의 평균 대기시간(분)과, 그 사람이 속한 버스 간격의 평균 길이(분, $t$ 큰 극한)는?
(c) 같은 노선의 '전형적' 간격 평균은?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\lambda t=3\cdot30=90$, $E[Y^2]=\mathrm{Var}(Y)+E[Y]^2=2500+10000=12500$. 평균 $90\cdot100=9000$, 분산 $90\cdot12500=1{,}125{,}000$, 표준편차 $\sqrt{1{,}125{,}000}\approx1060.66$. ($\lambda t\mathrm{Var}(Y)=225{,}000$이라고 하면 틀린다 — 청구 **건수**의 요동 $90\cdot100^2=900{,}000$이 더 크다.)

(b) 대기시간은 잔여수명 $R\sim\mathrm{Exp}(4/\text{h})$, 평균 $1/4$ h $=15$분. 그 사람이 속한 간격은 극한에서 $\mathrm{Gamma}(2,\lambda)$, 평균 $2/\lambda=30$분.

(c) 전형적 간격 평균은 $1/\lambda=15$분. 요점: 대기 $=$ 전형적 간격 **전체**(15분)이지 절반(7.5분)이 아니다.

```python
lam_t, EY, VarY = 3 * 30, 100.0, 2500.0
EY2 = VarY + EY**2
print(lam_t * EY, lam_t * EY2, math.sqrt(lam_t * EY2))   # 9000 1125000 1060.66
lam_bus = 4 / 60                                          # per minute
print(1 / lam_bus, 2 / lam_bus)                           # 15.0 30.0
```

</details>

### E2 (유도 후 시뮬레이션 검증)

$E[L(t)]=(2-e^{-\lambda t})/\lambda$를 (i) $E[A(t)]=\int_0^tP(A(t)>a)\,da$와 (ii) $R(t)\sim\mathrm{Exp}(\lambda)$로부터 유도하고, $\lambda=2$, $t=1.5$에서 시뮬레이션으로 검증하라. 추가: $t\to\infty$에서 $L(t)$의 밀도가 $\lambda^2xe^{-\lambda x}$임을 $A,R$의 독립성과 합성곱으로 보이고, $t=1.5$의 $L$ 히스토그램이 아직 이 극한과 얼마나 다른지 눈으로 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(i) $a<t$에서 $P(A(t)>a)=P\big((t-a,t]\text{에 도착 없음}\big)=e^{-\lambda a}$, $a\ge t$에서 $0$. 따라서 $E[A(t)]=\int_0^te^{-\lambda a}\,da=(1-e^{-\lambda t})/\lambda$. (ii) $\{R(t)>r\}=\{(t,t+r]\text{에 도착 없음}\}$이므로 $R(t)\sim\mathrm{Exp}(\lambda)$, $E[R(t)]=1/\lambda$. 합: $E[L(t)]=(2-e^{-\lambda t})/\lambda$. $\lambda=2$, $t=1.5$: $E[A]=(1-e^{-3})/2\approx0.4751$, $E[R]=0.5$, $E[L]=(2-e^{-3})/2\approx0.9751$.

**합성곱.** $A,R$ 독립, 극한에서 둘 다 $\mathrm{Exp}(\lambda)$이므로 $f_L(x)=\int_0^x\lambda e^{-\lambda a}\cdot\lambda e^{-\lambda(x-a)}\,da=\lambda^2e^{-\lambda x}\int_0^xda=\lambda^2xe^{-\lambda x}$. $t=1.5$에서는 $\lambda t=3$이라 $P(A=t)=e^{-3}\approx5\%$의 원자가 남아 있다. 정확한 분포는 $L(t)=\min(E_1,t)+E_2$에서 나온다: $x<t$에서는 $L(t)$의 분포가 극한과 **정확히 같다**(사건 $\{A+R\le x\}$는 $E_1<t$를 강제하므로). $x>t$에서 밀도는 $\lambda e^{-\lambda x}(1+\lambda t)$로, 극한 $\lambda^2xe^{-\lambda x}$의 $(1+\lambda t)/(\lambda x)$배다: $t<x<t+1/\lambda$ (여기서는 $1.5<x<2$)에서 극한보다 **두껍고** $x>2$에서 얇다. 원자 $e^{-3}$가 $x>1.5$로 밀려난 결과다. 아래 코드는 극한 밀도와 이 정확한 밀도를 함께 겹친다.

```python
lam2, t2 = 2.0, 1.5
paths2 = [poisson_arrivals(lam2, t2 + 15.0, rng) for _ in range(scale(50_000))]
A2, R2 = np.array([age_residual_at(p, t2) for p in paths2]).T
L2 = A2 + R2
exact_L2 = (2 - math.exp(-lam2 * t2)) / lam2
print(mc_estimate(A2), mc_estimate(R2), mc_estimate(L2), exact_L2)
assert_close(mc_estimate(L2), exact_L2, k=4, name="E[L(1.5)], lambda=2")
xs = np.linspace(0, 5, 300)
f_exact = np.where(xs < t2, lam2**2 * xs * np.exp(-lam2 * xs),
                   lam2 * np.exp(-lam2 * xs) * (1 + lam2 * t2))   # exact density of L(1.5)
ax = plot_hist_vs_pdf(L2, xs, stats.gamma(a=2, scale=1 / lam2).pdf(xs), bins=60,
                      label_pdf="Gamma(2, 2) limit")
ax.plot(xs, f_exact, "k-", lw=1.2, label="exact density of L(1.5)")
ax.axvline(t2, color="k", ls=":", label="t = 1.5 (age truncation)"); ax.legend(); plt.show()
```

</details>

### E3 (가정을 깨보기)

간격을 $\mathrm{Exp}(1)$ 대신 (a) 상수 1, (b) $U(0.5,1.5)$ (평균 1, 분산 1/12)로 바꾼 재생과정(`renewal_times(sampler, T, rng)`)에서 $t=20$을 덮는 간격 $L$과 잔여수명 $R$의 평균을 시뮬레이션하라. 검사 역설이 얼마나 남는가? 버스 대기시간은 간격 평균의 절반에 가까운가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 상수 간격: 도착이 $1,2,3,\dots$에 있으므로 $L\equiv1$ — 길이 편향이 있을 여지가 없다(간격이 모두 같으니). 격자 효과로 $R$은 $t$에 따라 결정론적이다: $t=20.3$이면 $A=0.3$, $R=0.7$. ($t=20$ 정각은 도착 시각 자체라 `age_residual_at`의 규약(마지막 도착 $\le t$)으로는 $A=0$, $R=1$이 나온다 — 우연속 규약이다.) 무작위 시각 $U(0,T)$로 보면 $E[R]=1/2$ — '간격의 절반'이 정확히 맞는 경우다.

(b) 정리 3: $\mu=1$, $E[X^2]=\mathrm{Var}+\mu^2=1/12+1=13/12$이므로 $E[L_\infty]=E[X^2]/\mu=13/12\approx1.0833$, $E[R_\infty]=E[X^2]/(2\mu)=13/24\approx0.5417$. 시뮬레이션(아래)이 이 값과 SE 안에서 일치한다. 지수 간격이었다면 $2$와 $1$이었다.

결론: 역설의 크기는 간격의 **변동계수**가 결정한다 — $E[L_\infty]/\mu=E[X^2]/\mu^2=1+\mathrm{CV}^2$. 지수분포($\mathrm{CV}=1$)가 '두 배'를, 규칙적 버스($\mathrm{CV}^2=1/12$)는 '거의 절반 대기'를 준다.

```python
t3, T3 = 20.0, 30.0
n3 = scale(20_000)
# (a) constant gaps
const = renewal_times(lambda n, g: np.ones(n), T3, rng)
print("constant gaps: (A, R) at t=20:", age_residual_at(const, t3), " at t=20.3:", age_residual_at(const, 20.3))
# (b) U(0.5, 1.5) gaps
sampler = lambda n, g: g.uniform(0.5, 1.5, n)
paths3 = [renewal_times(sampler, T3, rng) for _ in range(n3)]
A3, R3 = np.array([age_residual_at(p, t3) for p in paths3]).T
EX2 = 1 / 12 + 1.0
print("E[L(20)] =", mc_estimate(A3 + R3), " limit E[X^2]/mu =", EX2)        # ≈ 1.0833
print("E[R(20)] =", mc_estimate(R3), " limit E[X^2]/(2 mu) =", EX2 / 2)     # ≈ 0.5417
assert_close(mc_estimate(A3 + R3), EX2, k=4, name="E[L], uniform gaps")
assert_close(mc_estimate(R3), EX2 / 2, k=4, name="E[R], uniform gaps")
```

</details>

## 7. 정리

1. 복합 푸아송 $X(t)=\sum_{i\le N(t)}Y_i$: $E=\lambda tE[Y]$, $\mathrm{Var}=\lambda tE[Y^2]$ (총분산 법칙: 도착 수 요동 $+$ 크기 요동), mgf $\exp(\lambda t(M_Y-1))$.
2. 고정 $t$의 나이 $A(t)\sim\min(\mathrm{Exp}(\lambda),t)$, 잔여수명 $R(t)\sim\mathrm{Exp}(\lambda)$, 서로 독립 — 모두 '구간에 도착 없음' 사건의 확률로 한 줄 증명.
3. 검사 역설: $E[L(t)]=(2-e^{-\lambda t})/\lambda\to2/\lambda$, 극한 분포 $\mathrm{Gamma}(2,\lambda)$ = 지수분포의 길이 편향 버전. 간격을 '번호로' 고르면 $1/\lambda$, '시각으로' 고르면 $2/\lambda$.
4. 버스 대기 $=1/\lambda$ (간격 평균의 절반이 아님): 무작위 시각은 긴 간격에 걸리고, 무기억성으로 남은 절반이 다시 $\mathrm{Exp}(\lambda)$.
5. 일반 재생과정에서는 $E[L_\infty]=E[X^2]/\mu=\mu(1+\mathrm{CV}^2)$: 역설의 크기는 변동계수가 결정한다 (04b).

**trap 카드**
- Q: 버스가 평균 10분 간격(푸아송)으로 온다. 무작위로 정류장에 온 나는 평균 5분 기다린다 — 맞는가?
- A: 틀리다. 평균 10분을 기다린다. 무작위 시각은 긴 간격에 길이 비례로 더 자주 걸리므로 내가 속한 간격의 평균은 20분(길이 편향, $\mathrm{Gamma}(2,\lambda)$)이고, 무기억성으로 그 뒤쪽 절반의 기댓값은 다시 $1/\lambda=10$분이다. 규칙적인 버스라면 대기가 절반 근처($E[X^2]/2\mu$)로 줄어든다.

**다음 노트북: 03a.** 02a의 경쟁하는 지수 시계가 상태마다 하나씩 붙으면 연속시간 마르코프 사슬이 된다: 상태 $x$에서 비율 $q(x,y)$ 시계들이 경쟁하고, 첫 울림(비율 $q(x)=\sum_yq(x,y)$)으로 점프한다. 03a는 이 그림에서 생성원 $Q$와 질레스피 알고리즘을 만든다. 모듈 02의 푸아송 과정은 '상태가 1씩만 늘어나는 CTMC'라는 특수 경우로 다시 만난다.

`log/progress.md` 한 줄 템플릿: `- 2026-__-__ | 02d 복합 푸아송과 검사 역설 | 예측 6/6 중 __ 적중 | 막힌 곳: __ | 다음: 03a`